In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import librosa
from sklearn.pipeline import make_pipeline
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert
from math import log10
import parselmouth

In [2]:
csvTotale = pd.read_csv('./data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])

In [3]:
csvEval = pd.read_csv('./data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [4]:
(cross_val_score(HistGradientBoostingRegressor(categorical_features=csvTotale.drop(columns=['path', 'age']).dtypes == 'object'), 
                csvTotale.drop(columns=['path', 'age']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean(),
 cross_val_score(MLPRegressor(), 
                csvTotale.drop(columns=['path', 'age', 'ethnicity']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean())

# BASELINE = -7.24, -8.29

c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


(np.float64(-7.242994400990241), np.float64(-8.117801764116365))

In [5]:
audioDevelopment = {file: librosa.load('./data/allCleanAudio/'+file, sr=22050)[0] for file in csvTotale['path']}

In [6]:
def getMFCC(audio):
    return pd.Series(librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=14).mean(axis=1), 
                     index=[f'mfcc{i}' for i in range(14)])

In [7]:
def getRmsZcr(audio):
    return pd.Series([librosa.feature.rms(y=audio)[0].sum(), 
                      librosa.feature.zero_crossing_rate(y=audio)[0].mean(), librosa.feature.zero_crossing_rate(y=audio)[0].std()],
                        index=['rmsSum', 'zcrMean', 'zcrStd'])

In [8]:
def getSpectral(audio):
    return pd.Series([librosa.feature.spectral_centroid(y=audio)[0].mean(), librosa.feature.spectral_centroid(y=audio)[0].std(),
                        librosa.feature.spectral_bandwidth(y=audio)[0].mean(), librosa.feature.spectral_bandwidth(y=audio)[0].std(),
                        librosa.feature.spectral_contrast(y=audio)[0].mean(), librosa.feature.spectral_contrast(y=audio)[0].std(),
                        librosa.feature.spectral_flatness(y=audio)[0].mean(), librosa.feature.spectral_flatness(y=audio)[0].std(),
                        librosa.feature.spectral_rolloff(y=audio)[0].mean(), librosa.feature.spectral_rolloff(y=audio)[0].std()],
                        index=['spectralCentroidMean', 'spectralCentroidStd', 'spectralBandwidthMean', 'spectralBandwidthStd',
                                 'spectralContrastMean', 'spectralContrastStd', 'spectralFlatnessMean', 'spectralFlatnessStd',
                                    'spectralRolloffMean', 'spectralRolloffStd'])

In [9]:
def getSpectralEnergy(audio):
    S = librosa.stft(audio)
    freqs = librosa.fft_frequencies(sr=22050)
    return pd.Series([np.sum(np.abs(S[(freqs >= 250) & (freqs <= 650)])**2), np.sum(np.abs(S[(freqs >= 1000) & (freqs <= 8000)])**2)],
                        index=['spectralEnergy250-650', 'spectralEnergy1000-8000'])

In [10]:
def getSpectralRolloff(audio):
    return pd.Series([  librosa.feature.spectral_rolloff(y=audio, sr=22050, roll_percent=0.25).mean(),
                        librosa.feature.spectral_rolloff(y=audio, sr=22050, roll_percent=0.5).mean(),
                        librosa.feature.spectral_rolloff(y=audio, sr=22050, roll_percent=0.75).mean(),
                        librosa.feature.spectral_rolloff(y=audio, sr=22050, roll_percent=0.9).mean()],
                    index=['spectralRolloff25', 'spectralRolloff50', 'spectralRolloff75', 'spectralRolloff90'])

In [11]:
def getOtherSpectral(audio):    
    return  pd.Series([librosa.onset.onset_strength(y=audio, sr=2205).mean()], index=['onsetStrength'])

In [12]:
def getMathStats(audio):
    return pd.Series([np.mean(audio), np.std(audio), np.var(audio), skew(audio), kurtosis(audio)], 
                     index=['mean', 'std', 'var', 'skew', 'kurtosis'])

In [13]:
def getJitter(audio, end, start=0):
    point_process = parselmouth.praat.call(audio, "To PointProcess (periodic, cc)", 75, 600)  # pitch_floor=75, pitch_ceiling=600
    local_jitter =  parselmouth.praat.call(point_process, "Get jitter (local)", start, end, 0.0001, 0.02, 1.3)
    localabsolute_jitter =  parselmouth.praat.call(point_process, "Get jitter (local, absolute)", start, end, 0.0001, 0.02, 1.3)
    rap_jitter =    parselmouth.praat.call(point_process, "Get jitter (rap)", start, end, 0.0001, 0.02, 1.3)
    ppq5_jitter =   parselmouth.praat.call(point_process, "Get jitter (ppq5)", start, end, 0.0001, 0.02, 1.3)
    ddp_jitter =    parselmouth.praat.call(point_process, "Get jitter (ddp)", start, end, 0.0001, 0.02, 1.3)
    
    return pd.Series([local_jitter, localabsolute_jitter, rap_jitter, ppq5_jitter, ddp_jitter],
                     index=['localJitter', 'localAbsoluteJitter', 'rapJitter', 'ppq5Jitter', 'ddpJitter'])

In [14]:
def getHNRShimmer(sound, end, start=0):
    point_process = parselmouth.praat.call(sound, "To PointProcess (periodic, cc)", 75, 600)  # pitch_floor=75, pitch_ceiling=600
    local_shimmer = parselmouth.praat.call([sound, point_process], "Get shimmer (local)", start, end, 0.0001, 0.02, 1.3, 1.6)
    localdb_shimmer = parselmouth.praat.call([sound, point_process], "Get shimmer (local_dB)", start, end, 0.0001, 0.02, 1.3, 1.6)
    apq3_shimmer = parselmouth.praat.call([sound, point_process], "Get shimmer (apq3)", start, end, 0.0001, 0.02, 1.3, 1.6)
    apq5_shimmer = parselmouth.praat.call([sound, point_process], "Get shimmer (apq5)", start, end, 0.0001, 0.02, 1.3, 1.6)
    apq11_shimmer = parselmouth.praat.call([sound, point_process], "Get shimmer (apq11)", start, end, 0.0001, 0.02, 1.3, 1.6)
    dda_shimmer = parselmouth.praat.call([sound, point_process], "Get shimmer (dda)", start, end, 0.0001, 0.02, 1.3, 1.6)
    return pd.Series([local_shimmer, localdb_shimmer, apq3_shimmer, apq5_shimmer, apq11_shimmer, dda_shimmer],
                        index=['localShimmer', 'localdbShimmer', 'apq3Shimmer', 'apq5Shimmer', 'apq11Shimmer', 'ddaShimmer'])

In [15]:
def getPraatStats(sound):
        sound = parselmouth.Sound(sound)
        end = sound.total_duration
        return pd.concat([
            getJitter(sound, end=end),
            getHNRShimmer(sound, end=end),
            ], axis=0)

In [16]:
def getEnvelope(audio):
    temp = np.abs(hilbert(audio))
    return pd.Series([temp.mean(), temp.std()], index=['envelopeMean', 'envelopeStd'])

In [17]:
audioFeature = (pd.DataFrame({file: pd.concat([
                    getMFCC(audioDevelopment[file]), 
                    getRmsZcr(audioDevelopment[file]),
                    getSpectral(audioDevelopment[file]), 
                    getSpectralEnergy(audioDevelopment[file]),
                    getSpectralRolloff(audioDevelopment[file]), 
                    getOtherSpectral(audioDevelopment[file]),
                    getMathStats(audioDevelopment[file]),
                    getPraatStats(audioDevelopment[file]),
                    getEnvelope(audioDevelopment[file])
                    ], axis=0) for file in csvTotale['path']}).T) if 1 else pd.read_csv('audioFeature.csv', header=0, index_col=0)

In [18]:
totFeature = pd.concat([csvTotale.set_index(csvTotale['path']).drop(columns=['path', 'age']), audioFeature], axis=1)
totFeature = totFeature.drop(columns=['ethnicity'])
audioFeature = pd.concat([audioFeature, csvTotale.set_index(csvTotale['path'])['age']], axis=1)

In [19]:
# plt.figure(figsize=(40, 40))
# sns.heatmap(totFeature.corr('spearman'), annot=True, fmt=".2f", cmap='coolwarm');

In [20]:
cross_val_score(make_pipeline(StandardScaler(), HistGradientBoostingRegressor()),
                audioFeature.drop(columns=['age']), 
                audioFeature['age'], cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-7.218246765439565)

In [42]:
for col, corr in sorted(zip(audioFeature.columns, audioFeature.corr('spearman')['age']), key=lambda x: abs(x[1]), reverse=True):
    print(f'{col}: {corr:.2f}')

age: 1.00
rmsSum: 0.54
spectralBandwidthMean: -0.54
spectralEnergy250-650: 0.52
spectralFlatnessStd: 0.50
onsetStrength: -0.50
spectralFlatnessMean: 0.50
mean: 0.48
zcrStd: 0.48
spectralEnergy1000-8000: 0.48
zcrMean: 0.47
spectralRolloff25: 0.46
spectralCentroidStd: 0.45
mfcc6: -0.43
spectralContrastStd: 0.40
kurtosis: 0.38
spectralContrastMean: 0.37
mfcc1: -0.36
skew: -0.36
apq5Shimmer: 0.35
mfcc3: -0.35
apq3Shimmer: 0.34
ddaShimmer: 0.34
spectralRolloffStd: 0.33
spectralRolloff50: 0.33
mfcc0: -0.33
localShimmer: 0.32
apq11Shimmer: 0.31
mfcc2: -0.31
localdbShimmer: 0.30
mfcc9: -0.30
mfcc13: -0.29
spectralBandwidthStd: 0.28
spectralRolloff90: -0.27
mfcc4: -0.26
localJitter: 0.26
mfcc8: -0.26
mfcc11: -0.24
mfcc7: -0.22
ppq5Jitter: 0.20
localAbsoluteJitter: 0.20
spectralCentroidMean: 0.18
rapJitter: 0.17
ddpJitter: 0.17
spectralRolloffMean: -0.14
mfcc10: -0.14
envelopeStd: 0.12
envelopeMean: -0.08
mfcc5: 0.06
var: 0.05
std: 0.05
spectralRolloff75: 0.05
mfcc12: 0.01


In [22]:
audioEval = {file: librosa.load('./data/cleanAudioEval/'+file, sr=22050)[0] for file in csvEval['path']}

In [23]:
audioFeatureEval = (pd.DataFrame({file: pd.concat([
                    getMFCC(audioEval[file]), 
                    getRmsZcr(audioEval[file]),
                    getSpectral(audioEval[file]), 
                    getSpectralEnergy(audioEval[file]),
                    getSpectralRolloff(audioEval[file]), 
                    getOtherSpectral(audioEval[file]),
                    getMathStats(audioEval[file]),
                    getPraatStats(audioDevelopment[file]),
                    getEnvelope(audioDevelopment[file])
                    ], axis=0) for file in csvEval['path']}).T) if 1 else pd.read_csv('audioFeatureEval.csv', header=0, index_col=0)

In [96]:
cross_val_score(HistGradientBoostingRegressor(),
                audioFeature[(audioFeature['age']>= 20) & (audioFeature['age']<50)].drop(columns=['age']), 
                audioFeature[(audioFeature['age']>= 20) & (audioFeature['age']<50)]['age']
                , cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-5.277660609939662)